# 7. Comparison of Navier-Stokes and Reynolds Solvers

The implicit solver can also be run with the Reynolds equation, using a similar cavitation-complementarity approach as in the Navier-Stokes formulation. The Reynolds mode (`implicit_solver.equations.reynolds: True`) solves only the scalar, no-squeeze Reynolds equation for $p$.

Without inertia and in-plane shear, both formulations should yield the same steady state. We check this on three different cases below.

In [ ]:
from utils import load_template, run_reynolds_comparison, plot_reynolds_comparison_1d, plot_reynolds_comparison_2d, print_timing
from utils.topography import regen_twin_parabolic_slider_id

REF_DIR = '../../tests/ref_data'

## Inclined Slider

Linear converging gap with Dirichlet density BC. Note a slight offset due to discretizaion, which vanishes with increasing resolution.

<img src="./figures/Geometry_InclinedSlider.png"
     alt="Inclined slider geometry"
     width="250">

$\rightarrow$ **[YAML File](../../config_lib/validation/inclined_slider.yaml)**

In [ ]:
template = load_template('inclined_slider')
results = run_reynolds_comparison(template)
plot_reynolds_comparison_1d(results)
print_timing(results)

## Parabolic Slider 2D

Parabolic gap in $x$ and $y$ with Dirichlet pressure BC on all four sides, $N_x = N_y = 150$.

Here, the performance difference is clearly visible. While Navier-Stokes resolves 22 DOF per element (4 $\times p$, 9 $\times j_x$, 9 $\times j_y$), Reynolds only solves 4 $\times p$. With a direct LU solver, the computational cost grows superlinearly with the number of degrees of freedom ($\mathcal{O}(N^3)$ for dense matrices).

$\rightarrow$ **[YAML File](../../config_lib/validation/parabolic_slider_2d.yaml)**

In [ ]:
template = load_template('parabolic_slider_2d')
results = run_reynolds_comparison(template)
plot_reynolds_comparison_2d(results)
print_timing(results)

## Twin Parabolic Slider with Cavitation

Two identical parabolic sliders with cavitation. Both modes use UPG stabilization. The topography is set from Python before the run.

<img src="./figures/Geometry_TwinParabolicSliderIdentical.png"
     alt="Twin identical parabolic slider geometry"
     width="250">

$\rightarrow$ **[YAML File](../../config_lib/validation/twin_parabolic_slider_id.yaml)**

In [ ]:
template = load_template('twin_parabolic_slider_id')
results = run_reynolds_comparison(template, topo_fn=regen_twin_parabolic_slider_id)
plot_reynolds_comparison_1d(results, ref_csv=f'{REF_DIR}/ref_twin_parabolic_slider_id.csv', ref_label='Sahlin',
                            ref_p_scale=1e6, p_unit='MPa', p_divisor=1e6)
print_timing(results)

## Reynolds Equation

The Reynolds mode solves

$$
\frac{\partial (\rho h \theta')}{\partial t}
= \frac{\partial}{\partial x} \left( \frac{\rho h^3}{12 \mu} \frac{\partial p}{\partial x} \right)
+ \frac{\partial}{\partial y} \left( \frac{\rho h^3}{12 \mu} \frac{\partial p}{\partial y} \right)
- \frac{u_s}{2} \frac{\partial (\rho h \theta')}{\partial x}
- \frac{v_s}{2} \frac{\partial (\rho h \theta')}{\partial y}
$$

with the complementarity condition (Fischer-Burmeister form)

$$
p' + \theta - \sqrt{p'^2 + \theta^2} = 0, \qquad p' = \frac{p - p_\mathrm{cav}}{p_\mathrm{ref}}
$$

where $p$ is the pressure, $p_\mathrm{cav}$ the cavitation pressure and $p_\mathrm{ref}$ a reference pressure for normalization. $\rho(p)$ is the density given by the equation of state, $\mu$ the viscosity and $h$ the gap height. $u_s$ and $v_s$ are the sums of the wall velocities in $x$ and $y$. The cavity (void) fraction is $\theta$ and $\theta' = 1 - \theta$ the liquid fraction.